# NL Team Trends — Data Visualization Notebook

Comprehensive analysis and visualization of NL historical performance data
covering 1876 → 2025.

**Sources:** Baseball Reference, Retrosheet, SABR Lahman Database, Baseball Almanac

In [ ]:
# Load data
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # non-interactive backend
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (13, 7)

df        = pd.read_csv('data/nl_historical_data.csv')
franchise = pd.read_csv('data/franchise_summary.csv')
champs    = pd.read_csv('nl_champions.csv')
wins      = pd.read_csv('nl_team_wins.csv')
eras      = pd.read_csv('data/era_analysis.csv')

print(f'Historical data: {len(df)} rows, {df["season"].min()}–{df["season"].max()}')
print(f'Franchise summary: {len(franchise)} franchises')
print(f'Champions: {len(champs)} seasons')
print(f'Era analyses: {len(eras)} eras')

## 1. All-Time Franchise Win-Loss 

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top_franchises = franchise.sort_values('win_pct', ascending=False).head(8)
colors = ['#2D7D4F', '#3A6EA5', '#C8102E', '#FFB000', '#000000', '#0E3386', '#1D428A', '#183025']

axes[0].barh(top_franchises['franchise'], top_franchises['ws_titles'], color=colors[:len(top_franchises)])
axes[0].set_xlabel('World Series Titles')
axes[0].set_title('WS Titles by NL Franchise')

axes[1].scatter(franchise['seasons'], franchise['win_pct'] * 100, s=franchise['seasons']/3,
                c=colors, alpha=.8, edgecolors='white')
axes[1].set_xlabel('Seasons Played')
axes[1].set_ylabel('Win Percentage (%)')
axes[1].set_title('Win% vs Longevity')

for _, row in franchise.iterrows():
    name = row['franchise'].split()[-1]
    axes[1].annotate(name, (row['seasons'], row['win_pct']*100), fontsize=7)

plt.tight_layout()
plt.savefig('plots/franchise_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Era Win % Heatmap

In [ ]:
# Pivot table: Team × Era
pivot = df.pivot_table(values='win_pct', index='modern_name', columns='era', aggfunc='mean')

fig, ax = plt.subplots(figsize=(18, 8))
sns.heatmap(pivot, cmap='RdYlGn', center=.5, ax=ax,
            cbar_kws={'label': 'Win Percentage'}, linewidths=.5, annot=True, fmt='.2f')
ax.set_title('NL Franchise Win% by Era', fontsize=14)
ax.set_ylabel('Franchise')
plt.tight_layout()
plt.savefig('plots/era_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. 10-Year Rolling Win Average (Dynasty Detection)

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))
teams = ['St. Louis Cardinals','San Francisco Giants','Los Angeles Dodgers',
         'Chicago Cubs','Atlanta Braves','Pittsburgh Pirates']
palette = sns.color_palette('tab10', n_colors=len(teams))

for i, team in enumerate(teams):
    td = df[df['modern_name']==team].sort_values('season').copy()
    td['m5'] = td['win_pct'].rolling(5).mean()
    td['m10'] = td['win_pct'].rolling(10).mean()
    ax.plot(td['season'], td['m5'], label=f'{team} (5yr)', color=palette[i], alpha=.7, lw=2)
    ax.plot(td['season'], td['m10'], label=f'{team} (10yr)', color=palette[i], alpha=.3, lw=2, ls='--')

ax.set_xlabel('Season')
ax.set_ylabel('Rolling Win %')
ax.set_title('NL Franchise Dynasty Detection — 5-Year & 10-Year Rolling Win %')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
ax.axhline(.5, ls='-', color='black', alpha=.3)
plt.tight_layout()
plt.savefig('plots/dynasty_detection.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Championship Drought Tracker

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ws = champs[champs['Champion'] != '???'].dropna(subset=['Champion'])
for team in sorted(df['modern_name'].unique()):
    tw = ws[ws['Champion'] == team]
    if len(tw) > 1:
        gaps = tw['Year'].diff().iloc[1:]
        ax.scatter(range(len(gaps)), gaps, s=80, label=team.split()[-1], alpha=.8)
ax.set_xlabel('Championship Sequence')
ax.set_ylabel('Years Between Titles')
ax.set_title('NL Championship Droughts by Franchise')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('plots/championship_droughts.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Division Dominance Over Time

In [ ]:
df['decade'] = (df['season'] // 10) * 10
dw = df[df['division'] != 'N/A'].groupby(['decade', 'division']).size().reset_index(name='count')

fig, ax = plt.subplots(figsize=(14, 6))
sns.barplot(data=dw, x='decade', y='count', hue='division', ax=ax)
ax.set_xlabel('Decade')
ax.set_ylabel('Division Titles')
ax.set_title('NL Division Titles Won — by Decade & Division')
plt.tight_layout()
plt.savefig('plots/division_dominance.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Run Differential Trend Lines

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
top = ['St. Louis Cards', 'SF Giants', 'LA Dodgers', 'Chicago Cubs', 'Atlanta Braves']

for i, t in enumerate(top):
    d = df[df['modern_name'] == t].sort_values('season')
    axes[0].plot(d['season'], d['wins'] - d['losses'], lw=2, label=t)
axes[0].axhline(0, c='black', lw=.3, alpha=.3)
axes[0].set_xlabel('Season')
axes[0].set_title('Run Differential Over Time')
axes[0].legend(fontsize=8)

ed = df.groupby('era').agg({'win_pct':'mean', 'runs_scored':'mean', 'runs_allowed':'mean'}).reset_index()
x = np.arange(len(ed))
w = .25
axes[1].bar(x-w, ed['runs_scored'], w, label='RS', alpha=.8)
axes[1].bar(x, ed['runs_allowed'], w, label='RA', alpha=.8)
axes[1].bar(x+w, ed['win_pct']*100, w, label='Win%', alpha=.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(ed['era'], rotation=45, fontsize=8)
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.savefig('plots/era_comparison.png', dpi=150, bbox_inches='tight')
plt.show()